# Google Play: Dataset Preparation (notebook 1 of 2)
**Input:** `dataset_playstore1.csv` (GamesDB copy of the June 2021 Play Store scrape)
**Output:** `playstore_clean_dataset.csv` (one row per labelled game, no empty cells), a cleaning report, and a column policy. Notebook 2 trains on this file.

**What this notebook decides, in order**
1. Which rows are games (Sports is left out by default)
2. Which columns may be model inputs (only pre-launch fields) and which are label sources or post-launch
3. How each game's label is built (`1`, `0`, or left unlabelled)
4. How the data is cleaned: impossible values, empty cells, outliers, empty columns

**Cleaning rules (all counts are printed in a waterfall at the end)**

| Problem | Rule | Why |
|---|---|---|
| Contradictory rows (rating count above max installs, free but priced, not free but price 0, no developer) | Not labelled, so they leave the dataset | The values cannot all be true, and fixing them would be guessing |
| Paid game priced in a non-USD currency | Row dropped | The price cannot be compared, and there is no exchange rate in the file |
| Impossible numbers (size <= 0 or above 4 GB, Android version outside 1-15) | Set to missing, then handled as missing | Cannot be real |
| Missing **number** (size, minimum Android) | Filled with the **median**; a `..._was_missing` flag column records which cells were filled | Median is not pulled by extremes. The flag keeps the fact that the value was missing |
| Missing **category** (content rating) | Filled with the label `Unknown` | A category has no median. `Unknown` keeps the row and lets the model see that it was missing. The mode (most common value) would invent information |
| Missing **free text** | Not used in this module (the file has none) | If text existed, it would be left blank and flagged, never filled |
| Missing **identifier** (developer) | Row dropped | Needed to keep one studio's games together |
| **Outliers** (very large price or size) | **Capped** at the 99th percentile, not deleted | A $400 game or a 1.5 GB game is real. Deleting it would remove real games, and capping stops them dominating the model |
| Columns with no variation or over 95% empty | Dropped | They teach the model nothing |

**Known limit.** Medians and caps are calculated on all labelled games, not only on the training part, because the cleaned file is shared. The effect is tiny (one number per column), but state it.

## 1. Setup and settings

In [1]:
import os, re, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 70)
plt.rcParams['figure.figsize'] = (9, 4.5)

DATA_PATH = 'dataset_playstore1.csv'

INCLUDE_SPORTS = False      # Sports is both an app category and a game category on Google Play
MIN_RATINGS    = 50         # a game needs at least this many ratings to receive a label
LABEL_CUT      = 4.0        # average rating >= this -> well received (1); below -> not well received (0)

CAP_QUANTILE   = 0.99       # outliers are capped at this percentile
MAX_MISSING_FRAC = 0.95     # a column more empty than this is dropped
LOAD_ROWS = None            # e.g. 500_000 for a trial run

# GAME_CATS = {'action', 'adventure', 'arcade', 'board', 'card', 'casino', 'casual', 'educational',
#              'music', 'puzzle', 'racing', 'role playing', 'simulation', 'strategy', 'trivia', 'word'}
GAME_CATS = {'action', 'adventure', 'arcade', 'board', 'card', 'casino', 'casual', 'puzzle', 'racing', 'role playing', 'simulation', 'strategy', 'trivia', 'word'}
SPORTS_CAT = 'sports'
assert os.path.exists(DATA_PATH), f'{DATA_PATH} not found'
print('file:', DATA_PATH, '|', round(os.path.getsize(DATA_PATH) / 1e6), 'MB')

file: dataset_playstore1.csv | 676 MB


## 2. Load (games and Sports only; the e-mail column is never read)

In [2]:
USECOLS = ['App Name', 'App Id', 'Category', 'Rating', 'Rating Count', 'Installs', 'Minimum Installs',
           'Maximum Installs', 'Free', 'Price', 'Currency', 'Size', 'Minimum Android', 'Developer Id', 'Developer Website',
           'Released', 'Last Updated', 'Content Rating', 'Privacy Policy', 'Ad Supported', 'In App Purchases',
           'Editors Choice', 'Scraped Time']
head = pd.read_csv(DATA_PATH, nrows=1)
missing = [c for c in USECOLS if c not in head.columns]
assert not missing, f'Columns not found: {missing}'

keep_cats = GAME_CATS | {SPORTS_CAT}
parts, n_read = [], 0
for chunk in pd.read_csv(DATA_PATH, usecols=USECOLS, chunksize=300_000, nrows=LOAD_ROWS, low_memory=False):
    n_read += len(chunk)
    chunk = chunk[chunk['Category'].astype(str).str.strip().str.lower().isin(keep_cats)].copy()
    chunk['x_has_website'] = chunk['Developer Website'].notna()
    chunk['x_has_privacy'] = chunk['Privacy Policy'].notna()
    parts.append(chunk.drop(columns=['Developer Website', 'Privacy Policy']))
df = pd.concat(parts, ignore_index=True)
WATERFALL = [('rows in file', n_read), ('game or Sports categories', len(df))]
print(f'rows read: {n_read:,} | game or Sports rows kept: {len(df):,}')
print('dtypes of the two flags we rely on:', df['Ad Supported'].dtype, df['In App Purchases'].dtype)

rows read: 2,312,944 | game or Sports rows kept: 350,486
dtypes of the two flags we rely on: bool bool


## 3. Game scope (the Sports question)

In [3]:
is_sports = df['Category'].str.strip().str.lower().eq(SPORTS_CAT)
print(f'Sports rows: {is_sports.sum():,} of {len(df):,}')
print('\nMost-rated Sports apps:')
print(df[is_sports].sort_values('Rating Count', ascending=False)[['App Name', 'Rating Count']].head(25).to_string(index=False))
print('\nRandom Sports sample:')
print(df[is_sports].sample(min(25, int(is_sports.sum())), random_state=1)[['App Name']].to_string(index=False))

Sports rows: 47,483 of 350,486

Most-rated Sports apps:
                                 App Name  Rating Count
                              8 Ball Pool    21754025.0
                      Dream League Soccer    13387156.0
                              FIFA Soccer     7493304.0
                              Score! Hero     6869099.0
                       eFootball PES 2021     6782902.0
     Top Eleven 2021: Be a Soccer Manager     4751561.0
                   Carrom Pool: Disc Game     3240382.0
                 Dream League Soccer 2021     3239803.0
                           EA SPORTS UFC®     3002914.0
      World Cricket Championship 2 - WCC2     2974874.0
                             Bowling King     2937007.0
                 Pooking - Billiards City     2671414.0
               NBA LIVE Mobile Basketball     2390018.0
                          New Star Soccer     2287650.0
                             Soccer Stars     2247769.0
      Online Soccer Manager (OSM) - 20/21     20

In [4]:
if not INCLUDE_SPORTS:
    df = df[~is_sports].copy()
df = df.reset_index(drop=True)
WATERFALL.append((f'game categories (INCLUDE_SPORTS={INCLUDE_SPORTS})', len(df)))
print(f'games in scope: {len(df):,}')

games in scope: 303,003


## 4. Column policy: what may be a model input
The decisions are recorded in `playstore_column_policy.csv`. **Pre-launch** columns may be inputs. **Label** columns build the label and are never inputs. **Post-launch** columns are excluded.

Caveats: size, minimum Android, ads, in-app purchases, content rating and price are the **latest store listing** (June 2021), not guaranteed launch values. Release date and app name are **not** used as features.

In [5]:
POLICY = pd.DataFrame([
    ('Category',          'INPUT',               'Genre chosen by the developer.'),
    ('Content Rating',    'INPUT',               'Age rating declared by the developer.'),
    ('Free',              'INPUT (store state)', 'Listing value at scrape time.'),
    ('Price',             'INPUT (store state)', 'Listing value; used only when the currency is USD.'),
    ('Currency',          'USED to validate Price', 'Prices in other currencies are not comparable.'),
    ('Ad Supported',      'INPUT (monetisation design)', 'Boolean. Decided in the original design, but the file holds the latest listing.'),
    ('In App Purchases',  'INPUT (monetisation design)', 'Boolean. Same caveat as ads.'),
    ('Size',              'INPUT (store state)', 'Latest version size; successful games tend to grow.'),
    ('Minimum Android',   'INPUT (store state)', 'Latest version requirement.'),
    ('Developer Website', 'INPUT as yes/no flag', 'Whether a website is listed. URL dropped.'),
    ('Privacy Policy',    'INPUT as yes/no flag', 'Whether a policy is listed. URL dropped.'),
    ('Rating',            'LABEL SOURCE',        'Average stars after launch. Never an input.'),
    ('Rating Count',      'LABEL FILTER',        'Decides who is labelled. Never an input.'),
    ('Installs / Minimum Installs / Maximum Installs', 'POST-LAUNCH (excluded)', 'Popularity outcome.'),
    ('Last Updated',      'POST-LAUNCH (excluded)', 'Happens after launch.'),
    ('Editors Choice',    'POST-LAUNCH (excluded)', 'Editorial badge awarded after launch.'),
    ('Scraped Time',      'POST-LAUNCH (excluded)', 'Collection timestamp.'),
    ('Released',          'SPLIT ONLY',          'Orders games in time for the temporal test. Not a feature.'),
    ('Developer Id',      'GROUP ONLY',          'Keeps a studio in one fold. Stored as a hash because developer names can be personal names.'),
    ('App Name',          'NOT USED',            'Unique per app, so the raw name carries no learnable pattern. Not used.'),
    ('App Id',            'IDENTIFIER',          'Not a feature.'),
    ('Developer Email',   'NEVER LOADED',        'Personal data.'),
], columns=['column', 'role', 'reason'])
POLICY

,column,role,reason
0,Category,INPUT,Genre chosen by the developer.
1,Content Rating,INPUT,Age rating declared by the developer.
2,Free,INPUT (store state),Listing value at scrape time.
3,Price,INPUT (store state),Listing value; used only when the currency is USD.
4,Currency,USED to validate Price,Prices in other currencies are not comparable.
5,Ad Supported,INPUT (monetisation design),"Boolean. Decided in the original design, but the file holds the la..."
6,In App Purchases,INPUT (monetisation design),Boolean. Same caveat as ads.
7,Size,INPUT (store state),Latest version size; successful games tend to grow.
8,Minimum Android,INPUT (store state),Latest version requirement.
9,Developer Website,INPUT as yes/no flag,Whether a website is listed. URL dropped.


## 5. Parsing

In [6]:
def parse_size_mb(s):
    s = str(s).strip().replace(',', '')
    m = re.match(r'^([\d.]+)\s*([kKmMgG])$', s)
    if not m: return np.nan
    v, u = float(m.group(1)), m.group(2).lower()
    return v / 1024 if u == 'k' else (v * 1024 if u == 'g' else v)

def parse_min_android(s):
    m = re.match(r'^\s*(\d+)(?:\.(\d+))?(?:\.(\d+))?', str(s))
    if not m: return np.nan
    return int(m.group(1)) + (int(m.group(2)) / 10 if m.group(2) else 0) + (int(m.group(3)) / 100 if m.group(3) else 0)

def to_bool(s): return s.astype(str).str.lower().isin(['true', '1'])

d = df
d['x_rating']       = pd.to_numeric(d['Rating'], errors='coerce')
d['x_rating_count'] = pd.to_numeric(d['Rating Count'], errors='coerce')
d['x_max_inst']     = pd.to_numeric(d['Maximum Installs'], errors='coerce')
d['x_min_inst']     = pd.to_numeric(d['Minimum Installs'], errors='coerce')
d['x_price_raw']    = pd.to_numeric(d['Price'], errors='coerce').fillna(0)
d['x_currency']     = d['Currency'].fillna('Unknown').astype(str).str.upper()
d['x_free']         = to_bool(d['Free'])
d['x_price']        = np.where((d['x_price_raw'] > 0) & (d['x_currency'] != 'USD'), np.nan, d['x_price_raw'])
d['x_ads']          = to_bool(d['Ad Supported'])
d['x_iap']          = to_bool(d['In App Purchases'])
d['x_editors']      = to_bool(d['Editors Choice'])
d['x_size_mb']      = d['Size'].map(parse_size_mb)
d['x_min_android']  = d['Minimum Android'].map(parse_min_android)
d['x_released']     = pd.to_datetime(d['Released'], format='%b %d, %Y', errors='coerce')
d['x_updated']      = pd.to_datetime(d['Last Updated'], format='%b %d, %Y', errors='coerce')
d['x_release_year'] = d['x_released'].dt.year
paid = d['x_price_raw'] > 0
print(f"paid games: {paid.sum():,} | paid and not in USD: {(paid & (d['x_currency'] != 'USD')).sum():,}")
print('Ad Supported and In App Purchases are boolean:', set(d['x_ads'].unique()) <= {True, False}, set(d['x_iap'].unique()) <= {True, False})
print('size parsed:', f"{d['x_size_mb'].notna().mean()*100:.1f}%", '| min android parsed:', f"{d['x_min_android'].notna().mean()*100:.1f}%", '| released parsed:', f"{d['x_released'].notna().mean()*100:.1f}%")

paid games: 7,306 | paid and not in USD: 0
Ad Supported and In App Purchases are boolean: True True
size parsed: 93.9% | min android parsed: 96.3% | released parsed: 96.9%


## 6. Data-quality flags

In [7]:
flags = pd.DataFrame(index=d.index)
flags['rating_count_gt_max_installs'] = d['x_rating_count'] > d['x_max_inst']
flags['not_free_but_price_zero']      = (~d['x_free']) & (d['x_price_raw'] == 0)
flags['free_but_price_positive']      = d['x_free'] & (d['x_price_raw'] > 0)
flags['updated_before_released']      = d['x_updated'] < d['x_released']
flags['released_missing']             = d['x_released'].isna()
flags['no_developer']                 = d['Developer Id'].isna()
display(pd.DataFrame({'rows': flags.sum(), '%': (flags.mean() * 100).round(2)}))
d['x_bad_row'] = flags[['rating_count_gt_max_installs', 'not_free_but_price_zero', 'free_but_price_positive', 'no_developer']].any(axis=1)
print('contradictory rows (will not be labelled):', int(d['x_bad_row'].sum()))

,rows,%
rating_count_gt_max_installs,112,0.04
not_free_but_price_zero,84,0.03
free_but_price_positive,0,0.00
updated_before_released,1158,0.38
released_missing,9539,3.15
no_developer,0,0.00


contradictory rows (will not be labelled): 196


## 7. Label design and row-level label assignment

| Row situation | `label` | `label_status` |
|---|---|---|
| rating count >= `MIN_RATINGS` and rating >= `LABEL_CUT` | **1** (well received) | `positive` |
| rating count >= `MIN_RATINGS` and rating < `LABEL_CUT` | **0** (not well received) | `negative` |
| rating 0 and count 0 | blank | `unrated` |
| 0 < rating count < `MIN_RATINGS` | blank | `below_floor` |
| contradictory row (section 6) | blank | `bad_row` |

Unrated and below-floor games are **not** labelled 0. Having no ratings says nothing about whether players liked the game.

**Why these defaults.** `MIN_RATINGS = 50` and `LABEL_CUT = 4.0` follow the provisional rule already in the evidence log (Decisions 6 and 13): a floor so the average rests on enough ratings, and a fixed cut so "well received" means the same thing in every category. 4.0 stars is a convention, not a derived value, so the tables below show what other choices would give. Change the two settings in section 1 if you prefer another.

In [8]:
rated = d[(d['x_rating'] > 0) & (d['x_rating_count'] > 0) & ~d['x_bad_row']]
floors = [10, 25, 50, 100, 250, 500]
cuts = [3.5, 3.8, 4.0, 4.2, 4.4, 4.5]
n_tbl, p_tbl = {}, {}
for fl in floors:
    sub = rated[rated['x_rating_count'] >= fl]
    n_tbl[fl] = len(sub)
    p_tbl[fl] = {c: round((sub['x_rating'] >= c).mean() * 100, 1) if len(sub) else np.nan for c in cuts}
print(f'games in scope: {len(d):,} | rated (rating>0, no contradiction): {len(rated):,}')
print('\nLabelled games by rating-count floor:')
print(pd.Series(n_tbl, name='games').to_frame().T.to_string())
print('\n% labelled positive (rating >= cut), by floor (rows) and cut (columns):')
display(pd.DataFrame(p_tbl).T.rename_axis('floor').rename(columns=lambda c: f'>={c}'))

games in scope: 303,003 | rated (rating>0, no contradiction): 196,200

Labelled games by rating-count floor:
          10      25      50     100    250    500
games  164061  125264  102215  83977  63006  50880

% labelled positive (rating >= cut), by floor (rows) and cut (columns):


,>=3.5,>=3.8,>=4.0,>=4.2,>=4.4,>=4.5
floor,,,,,,
10,87.2,74.9,62.4,48.1,33.1,26.2
25,87.4,74.1,60.4,44.3,28.0,20.7
50,87.7,74.0,59.8,42.8,25.5,17.8
100,88.1,74.5,60.1,42.5,24.3,16.4
250,89.4,76.5,61.9,43.6,24.5,16.1
500,90.5,78.5,64.2,45.6,25.6,16.7


In [9]:
d['label'] = np.nan
d['label_status'] = ''
unrated  = (d['x_rating'].fillna(0) == 0) | (d['x_rating_count'].fillna(0) == 0)
bad      = d['x_bad_row']
below    = ~unrated & ~bad & (d['x_rating_count'] < MIN_RATINGS)
labelled = ~unrated & ~bad & (d['x_rating_count'] >= MIN_RATINGS)
d.loc[bad, 'label_status'] = 'bad_row'
d.loc[unrated & ~bad, 'label_status'] = 'unrated'
d.loc[below, 'label_status'] = 'below_floor'
d.loc[labelled & (d['x_rating'] >= LABEL_CUT), ['label', 'label_status']] = [1, 'positive']
d.loc[labelled & (d['x_rating'] <  LABEL_CUT), ['label', 'label_status']] = [0, 'negative']

print(f'MIN_RATINGS = {MIN_RATINGS}, LABEL_CUT = {LABEL_CUT}\n')
print(d['label_status'].value_counts().to_string())
L = d[d['label'].notna()].copy()
L['label'] = L['label'].astype(int)
WATERFALL.append((f'labelled (>= {MIN_RATINGS} ratings, no contradiction)', len(L)))
print(f'\nlabelled games: {len(L):,} | well received {L["label"].mean()*100:.1f}% | not well received {100-L["label"].mean()*100:.1f}%')

MIN_RATINGS = 50, LABEL_CUT = 4.0

label_status
unrated        106607
below_floor     93985
positive        61085
negative        41130
bad_row           196

labelled games: 102,215 | well received 59.8% | not well received 40.2%


In [10]:
# Evidence for excluding post-launch columns: how well each predicts the label on its own
cands = {'rating count': L['x_rating_count'], 'max installs': L['x_max_inst'], 'min installs': L['x_min_inst'],
         'editors choice': L['x_editors'].astype(int), 'days since last update': (pd.Timestamp('2021-06-16') - L['x_updated']).dt.days}
rows = []
for k, s in cands.items():
    m_ = s.notna()
    if m_.sum() > 100 and L.loc[m_, 'label'].nunique() == 2:
        rows.append((k, int(m_.sum()), round(roc_auc_score(L.loc[m_, 'label'], s[m_]), 3)))
display(pd.DataFrame(rows, columns=['excluded column', 'rows', 'AUC alone vs label']))
print('Distance from 0.5 shows how much outcome information a column carries. None of these is given to the model.')

,excluded column,rows,AUC alone vs label
0,rating count,102215,0.569
1,max installs,102215,0.485
2,min installs,102215,0.486
3,editors choice,102215,0.503
4,days since last update,102215,0.424


Distance from 0.5 shows how much outcome information a column carries. None of these is given to the model.


## 8. Cleaning
Every step logs how many rows or cells it touched.

In [11]:
C = L.copy()
CLEAN_LOG = []   # (step, rows_before, rows_after, cells_changed, note)

def log(step, before, after, changed, note=''):
    CLEAN_LOG.append((step, before, after, changed, note))

# 1. paid games priced in a non-USD currency: the price cannot be compared -> drop rows
n0 = len(C); m = C['x_price'].isna()
C = C[~m].copy(); log('drop paid games with non-USD price', n0, len(C), 0, 'price unknown without exchange rates')

# 2. impossible numbers -> missing (then treated as missing in step 4)
bad_size = (C['x_size_mb'] <= 0) | (C['x_size_mb'] > 4096)
bad_and  = (C['x_min_android'] < 1) | (C['x_min_android'] > 15)
C.loc[bad_size, 'x_size_mb'] = np.nan; C.loc[bad_and, 'x_min_android'] = np.nan
log('impossible size / Android version set to missing', len(C), len(C), int(bad_size.sum() + bad_and.sum()), 'size <= 0 or > 4 GB; Android outside 1-15')

# 3. categorical missing -> 'Unknown' (a category has no median; the mode would invent information)
C['content_rating'] = C['Content Rating'].fillna('Unknown').astype(str).str.strip()
C['category'] = C['Category'].astype(str).str.strip().str.title()
n_unknown = int((C['Content Rating'].isna()).sum())
log("missing content rating -> 'Unknown'", len(C), len(C), n_unknown)

# 4. missing numbers -> median, with a flag column recording which cells were filled
fill_report = {}
for col, flag in [('x_size_mb', 'size_was_missing'), ('x_min_android', 'min_android_was_missing')]:
    C[flag] = C[col].isna().astype(int)
    med = float(C[col].median())
    fill_report[col] = {'median_used': round(med, 3), 'cells_filled': int(C[flag].sum()), 'pct_filled': round(C[flag].mean() * 100, 2)}
    C[col] = C[col].fillna(med)
log('missing size / Android version -> median (+ flag columns)', len(C), len(C), int(sum(v['cells_filled'] for v in fill_report.values())), json.dumps(fill_report))

# 5. outliers: cap at the 99th percentile (price among paid games only), do not delete
cap_report = {}
paid_prices = C.loc[C['x_price'] > 0, 'x_price']
caps = {'x_price': float(paid_prices.quantile(CAP_QUANTILE)) if len(paid_prices) else None,
        'x_size_mb': float(C['x_size_mb'].quantile(CAP_QUANTILE))}
for col, cap in caps.items():
    if cap is None: continue
    n_cap = int((C[col] > cap).sum())
    C[col] = C[col].clip(upper=cap)
    cap_report[col] = {'cap': round(cap, 2), 'cells_capped': n_cap}
log(f'outliers capped at the {int(CAP_QUANTILE*100)}th percentile', len(C), len(C), int(sum(v['cells_capped'] for v in cap_report.values())), json.dumps(cap_report))

# 6. developer: pseudonymise; missing developer rows were already removed in section 6
C['developer_hash'] = C['Developer Id'].astype(str).map(lambda t: hashlib.sha256(t.encode()).hexdigest()[:12])

# 7. assemble the clean frame
clean = pd.DataFrame({
    'app_id': C['App Id'].values, 'developer_hash': C['developer_hash'].values,
    'category': C['category'].values, 'content_rating': C['content_rating'].values,
    'free': C['x_free'].astype(int).values, 'price_usd': C['x_price'].values,
    'ad_supported': C['x_ads'].astype(int).values, 'in_app_purchases': C['x_iap'].astype(int).values,
    'size_mb': C['x_size_mb'].values, 'min_android': C['x_min_android'].values,
    'has_website': C['x_has_website'].astype(int).values, 'has_privacy': C['x_has_privacy'].astype(int).values,
    'size_was_missing': C['size_was_missing'].values, 'min_android_was_missing': C['min_android_was_missing'].values,
    'release_date': C['x_released'].dt.strftime('%Y-%m-%d').values, 'release_year': C['x_release_year'].values,
    'label': C['label'].astype(int).values,
    'audit_rating': C['x_rating'].values, 'audit_rating_count': C['x_rating_count'].values,   # audit only: never inputs
})

# 8. drop empty or constant columns (among the input columns)
INPUT_COLS = ['category', 'content_rating', 'free', 'price_usd', 'ad_supported', 'in_app_purchases', 'size_mb',
              'min_android', 'has_website', 'has_privacy', 'size_was_missing', 'min_android_was_missing']
dropped = []
for c in INPUT_COLS:
    if clean[c].isna().mean() > MAX_MISSING_FRAC or clean[c].nunique(dropna=True) <= 1:
        dropped.append(c)
clean = clean.drop(columns=dropped)
log('drop empty or constant input columns', len(clean), len(clean), 0, str(dropped) if dropped else 'none dropped')

pd.DataFrame(CLEAN_LOG, columns=['step', 'rows_before', 'rows_after', 'cells_changed', 'note'])

,step,rows_before,rows_after,cells_changed,note
0,drop paid games with non-USD price,102215,102215,0,price unknown without exchange rates
1,impossible size / Android version set to missing,102215,102215,0,size <= 0 or > 4 GB; Android outside 1-15
2,missing content rating -> 'Unknown',102215,102215,0,
3,missing size / Android version -> median (+ flag columns),102215,102215,9357,"{""x_size_mb"": {""median_used"": 33.0, ""cells_filled"": 6895, ""pct_fil..."
4,outliers capped at the 99th percentile,102215,102215,1021,"{""x_price"": {""cap"": 17.94, ""cells_capped"": 39}, ""x_size_mb"": {""cap..."
5,drop empty or constant input columns,102215,102215,0,none dropped


In [12]:
# Final checks
input_present = [c for c in INPUT_COLS if c in clean.columns]
empty_cells = int(clean[input_present + ['label']].isna().sum().sum())
print('empty cells in input columns and label:', empty_cells)
assert empty_cells == 0, 'There are still empty cells'
assert clean['app_id'].is_unique, 'Duplicate app ids'
assert set(clean['label'].unique()) <= {0, 1}
assert clean['price_usd'].min() >= 0 and clean['size_mb'].min() > 0
print('rows:', len(clean), '| well received:', f"{clean['label'].mean()*100:.1f}%", '| inputs:', len(input_present))
WATERFALL.append(('clean dataset (final)', len(clean)))
display(pd.DataFrame(WATERFALL, columns=['step', 'rows']))
display(clean[input_present].describe(include='all').T.round(3))

empty cells in input columns and label: 0
rows: 102215 | well received: 59.8% | inputs: 12


,step,rows
0,rows in file,2312944
1,game or Sports categories,350486
2,game categories (INCLUDE_SPORTS=False),303003
3,"labelled (>= 50 ratings, no contradiction)",102215
4,clean dataset (final),102215


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
category,102215,14,Puzzle,15062,NaN,NaN,NaN,NaN,NaN,NaN,NaN
content_rating,102215,6,Everyone,68309,NaN,NaN,NaN,NaN,NaN,NaN,NaN
free,102215.0,NaN,NaN,NaN,0.96252,0.189935,0.0,1.0,1.0,1.0,1.0
price_usd,102215.0,NaN,NaN,NaN,0.137056,0.903055,0.0,0.0,0.0,0.0,17.945
ad_supported,102215.0,NaN,NaN,NaN,0.852478,0.354628,0.0,1.0,1.0,1.0,1.0
in_app_purchases,102215.0,NaN,NaN,NaN,0.431434,0.495279,0.0,0.0,0.0,1.0,1.0
size_mb,102215.0,NaN,NaN,NaN,39.465336,29.308474,0.014648,18.0,33.0,53.0,145.0
min_android,102215.0,NaN,NaN,NaN,4.048557,0.906724,1.0,4.03,4.1,4.4,8.0
has_website,102215.0,NaN,NaN,NaN,0.751328,0.432245,0.0,1.0,1.0,1.0,1.0
has_privacy,102215.0,NaN,NaN,NaN,0.876496,0.329017,0.0,1.0,1.0,1.0,1.0


## 9. Export

In [13]:
clean.to_csv('playstore_clean_dataset.csv', index=False)
POLICY.to_csv('playstore_column_policy.csv', index=False)
report = {'settings': {'INCLUDE_SPORTS': INCLUDE_SPORTS, 'MIN_RATINGS': MIN_RATINGS, 'LABEL_CUT': LABEL_CUT, 'CAP_QUANTILE': CAP_QUANTILE},
          'waterfall': WATERFALL, 'cleaning_steps': [dict(zip(['step', 'rows_before', 'rows_after', 'cells_changed', 'note'], r)) for r in CLEAN_LOG],
          'fill_medians': fill_report, 'caps': cap_report, 'dropped_columns': dropped,
          'labelled_rows': int(len(clean)), 'pct_well_received': round(float(clean['label'].mean() * 100), 1)}
json.dump(report, open('playstore_cleaning_report.json', 'w'), indent=2, default=str)
print('wrote playstore_clean_dataset.csv, playstore_column_policy.csv, playstore_cleaning_report.json')

wrote playstore_clean_dataset.csv, playstore_column_policy.csv, playstore_cleaning_report.json
